Step 1: Import libraries and load the datasets

I import the tools needed for data preparation, model training, cross-validation, hyperparameter tuning, and evaluation. Then I load the classification and regression CSV files.

In [30]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_validate, GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    confusion_matrix, f1_score,
    precision_score, recall_score, mean_absolute_error,
    mean_squared_error, r2_score
)

clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")

display(clf_df.head(), reg_df.head())
print("Classification shape:", clf_df.shape)
print("Regression shape:", reg_df.shape)

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


Classification shape: (900, 7)
Regression shape: (800, 7)


Step 2: Prepare the classification data

I separate the input features (`X_clf`) from the target (`y_clf`). I remove `ticket_id` because it is only an identifier. I split the data into 80% training and 20% testing. `stratify=y_clf` keeps the class proportions similar in both sets.

In [19]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

Step 3: Preprocess the classification data

I identify numerical and categorical columns because they need different preprocessing.

- Numerical: fill missing values with the median, then scale the values.
- Categorical: fill missing values with the most frequent category, then convert categories into numeric columns using one-hot encoding.

`ColumnTransformer` applies the correct steps to each column type.

In [20]:
numeric_clf = X_clf.select_dtypes(include=np.number).columns.tolist()
categorical_clf = X_clf.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_clf = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_clf),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_clf)
    ]
)

Step 4: Compare classification models using cross-validation

I compare Logistic Regression, k-NN, and Decision Tree using 5-fold Stratified Cross-Validation.

`scoring="f1"` calculates the F1 score for each fold. I use the mean CV F1 to compare the models. The model with the highest mean F1 performs best on average across the five folds.

In [21]:
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "k-NN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

classification_results = []
for name, estimator in classifiers.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_clf),
        ("model", estimator)
    ])
    scores = cross_validate(
        pipe, X_train_clf, y_train_clf,
        cv=cv_clf, scoring="f1", return_train_score=False
    )
    classification_results.append({
        "Model": name,
        "Mean CV F1": scores["test_score"].mean(),
        "Std CV F1": scores["test_score"].std()
    })

classification_results_df = pd.DataFrame(classification_results).sort_values(
    "Mean CV F1", ascending=False
)
display(classification_results_df)

,Model,Mean CV F1,Std CV F1
0,Logistic Regression,0.392651,0.062747
1,k-NN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


Step 5: Tune the classification models

I use `GridSearchCV` to test different hyperparameter settings with the same 5-fold cross-validation. It selects the settings with the highest mean F1 score.

- Logistic Regression: `C`
- k-NN: `n_neighbors` and `weights`
- Decision Tree: `max_depth` and `min_samples_split`

I select the best classification model by comparing the tuned models' mean CV F1 scores.

In [22]:
classification_searches = {
    "Logistic Regression": (
        Pipeline([
            ("preprocessor", preprocessor_clf),
            ("model", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        {"model__C": [0.01, 0.1, 1, 10, 100]}
    ),
    "k-NN": (
        Pipeline([
            ("preprocessor", preprocessor_clf),
            ("model", KNeighborsClassifier())
        ]),
        {"model__n_neighbors": [3, 5, 7, 9], "model__weights": ["uniform", "distance"]}
    ),
    "Decision Tree": (
        Pipeline([
            ("preprocessor", preprocessor_clf),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        {"model__max_depth": [None, 3, 5, 8, 12], "model__min_samples_split": [2, 5, 10]}
    )
}

classification_search_results = []
classification_best_models = {}
classification_search_objects = {}

for name, (pipeline, parameters) in classification_searches.items():
    search = GridSearchCV(
        pipeline, param_grid=parameters, cv=cv_clf,
        scoring="f1", n_jobs=-1, refit=True
    )
    search.fit(X_train_clf, y_train_clf)
    classification_best_models[name] = search.best_estimator_
    classification_search_objects[name] = search
    classification_search_results.append({
        "Model": name,
        "Best CV F1": search.best_score_,
        "Best hyperparameters": search.best_params_
    })

tuned_classification_results = pd.DataFrame(classification_search_results).sort_values(
    "Best CV F1", ascending=False
)
display(tuned_classification_results)


best_clf_name = tuned_classification_results.iloc[0]["Model"]
best_clf = classification_best_models[best_clf_name]
print("Selected classification model:", best_clf_name)

,Model,Best CV F1,Best hyperparameters
0,Logistic Regression,0.422593,{'model__C': 10}
1,k-NN,0.390646,"{'model__n_neighbors': 3, 'model__weights': 'u..."
2,Decision Tree,0.387218,"{'model__max_depth': 12, 'model__min_samples_s..."


Selected classification model: Logistic Regression


Logistic Regression: compare C before and after tuning

Logistic Regression starts with the default C = 1. I compare its mean CV F1 at the default C with the score after tuning.
What C means:
- Small C = strong regularization. It restricts the model more, so the model is less flexible and may underfit.
- Large C = weak regularization. It restricts the model less, so the model is more flexible and may overfit.

C is the inverse of regularization strength. GridSearchCV chooses the C value that gives the best mean CV F1 from the tested values.

In [23]:
logistic_search = classification_search_objects['Logistic Regression']
logistic_cv_results = pd.DataFrame(logistic_search.cv_results_)
default_c_row = logistic_cv_results[
    logistic_cv_results['param_model__C'].astype(float) == 1.0
].iloc[0]

c_comparison = pd.DataFrame([
    {
        'Stage': 'Before tuning (default)',
        'C value': 1.0,
        'Mean CV F1': default_c_row['mean_test_score']
    },
    {
        'Stage': 'After tuning (best)',
        'C value': float(logistic_search.best_params_['model__C']),
        'Mean CV F1': logistic_search.best_score_
    }
])

display(c_comparison)

,Stage,C value,Mean CV F1
0,Before tuning (default),1.0,0.392651
1,After tuning (best),10.0,0.422593


Step 6: Evaluate the best classification model

I use the model selected by cross-validation to predict the unseen test data. I evaluate it using precision, recall, and F1 score. I also show the confusion matrix.

- Precision: Of the cases predicted as escalated, how many were actually escalated?
- Recall: Of all cases that were actually escalated, how many did the model find?
- F1: the balance between precision and recall.

The best model is the one with the highest tuned mean CV F1, not the one chosen by looking at the test results.

In [24]:
y_pred_clf = best_clf.predict(X_test_clf)

classification_test_metrics = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1 score"],
    "Test score": [
        precision_score(y_test_clf, y_pred_clf, zero_division=0),
        recall_score(y_test_clf, y_pred_clf, zero_division=0),
        f1_score(y_test_clf, y_pred_clf)
    ]
})
display(classification_test_metrics)
display(pd.DataFrame(
    confusion_matrix(y_test_clf, y_pred_clf),
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

,Metric,Test score
0,Precision,0.615385
1,Recall,0.210526
2,F1 score,0.313725


,Predicted 0,Predicted 1
Actual 0,137,5
Actual 1,30,8


Step 7: Prepare the regression data

I separate the input features (`X_reg`) from the target (`y_reg`), which is `resolution_time_hours`. I remove `ticket_id` and split the data into 80% training and 20% testing.

In [25]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

Step 8: Preprocess the regression data

I use the same preprocessing approach as classification: median imputation and scaling for numerical columns, and most-frequent imputation and one-hot encoding for categorical columns.

In [26]:
numeric_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
categorical_reg = X_reg.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_reg = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_reg),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_reg)
    ]
)

Step 9: Compare regression models using cross-validation

I compare Linear Regression, Ridge Regression, and Decision Tree Regressor using 5-fold K-Fold cross-validation.

I use MAE as the scoring metric. A lower mean CV MAE means the model's predictions are closer to the actual values on average.

In [27]:
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42)
}

regression_results = []
for name, estimator in regressors.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("model", estimator)
    ])
    scores = cross_validate(
        pipe, X_train_reg, y_train_reg,
        cv=cv_reg, scoring="neg_mean_absolute_error",
        return_train_score=False
    )
    fold_mae = -scores["test_score"]
    regression_results.append({
        "Model": name,
        "Mean CV MAE": fold_mae.mean(),
        "Std CV MAE": fold_mae.std()
    })

regression_results_df = pd.DataFrame(regression_results).sort_values(
    "Mean CV MAE", ascending=True
)
display(regression_results_df)

,Model,Mean CV MAE,Std CV MAE
1,Ridge Regression,2.035127,0.171404
0,Linear Regression,2.035134,0.171650
2,Decision Tree Regressor,3.239097,0.053807


Step 10: Tune the regression models

I use `GridSearchCV` to test the selected hyperparameter values.

- Ridge Regression: `alpha` controls regularization strength.
- Decision Tree Regressor: `max_depth` and `min_samples_split` control the tree.
- Linear Regression: I keep it as the baseline because there is no hyperparameter grid for it in this exercise.

I select the regression model with the lowest mean CV MAE.

In [28]:
regression_searches = {
    "Linear Regression": (
        Pipeline([
            ("preprocessor", preprocessor_reg),
            ("model", LinearRegression())
        ]),
        [{}]
    ),
    "Ridge Regression": (
        Pipeline([
            ("preprocessor", preprocessor_reg),
            ("model", Ridge())
        ]),
        {"model__alpha": [0.01, 0.1, 1, 10, 100]}
    ),
    "Decision Tree Regressor": (
        Pipeline([
            ("preprocessor", preprocessor_reg),
            ("model", DecisionTreeRegressor(random_state=42))
        ]),
        {"model__max_depth": [None, 3, 5, 8, 12],
         "model__min_samples_split": [2, 5, 10]}
    )
}

regression_search_results = []
regression_best_models = {}

for name, (pipeline, parameters) in regression_searches.items():
    search = GridSearchCV(
        pipeline,
        param_grid=parameters,
        cv=cv_reg,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train_reg, y_train_reg)

    regression_best_models[name] = search.best_estimator_
    regression_search_results.append({
        "Model": name,
        "Best CV MAE": -search.best_score_,
        "Best hyperparameters": search.best_params_
    })

tuned_regression_results = pd.DataFrame(regression_search_results).sort_values(
    "Best CV MAE", ascending=True
)
display(tuned_regression_results)

# Select the model with the lowest cross-validation MAE.
best_reg_name = tuned_regression_results.iloc[0]["Model"]
best_reg = regression_best_models[best_reg_name]

print("Best regression model based on cross-validation:", best_reg_name)
print("Lowest mean CV MAE:", round(tuned_regression_results.iloc[0]["Best CV MAE"], 4))

,Model,Best CV MAE,Best hyperparameters
1,Ridge Regression,2.035118,{'model__alpha': 0.1}
0,Linear Regression,2.035134,{}
2,Decision Tree Regressor,2.434134,"{'model__max_depth': 5, 'model__min_samples_sp..."


Best regression model based on cross-validation: Ridge Regression
Lowest mean CV MAE: 2.0351


Step 11: Evaluate the best regression model

I use the selected model to predict resolution time for the unseen test data.

- MAE: average absolute prediction error.
- RMSE: penalizes large prediction errors more strongly.
- R²: measures how much variation in resolution time the model explains.

Lower MAE and RMSE are better. An R² closer to 1 is generally better.

In [29]:
y_pred_reg = best_reg.predict(X_test_reg)

regression_test_metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Test score": [
        mean_absolute_error(y_test_reg, y_pred_reg),
        mean_squared_error(y_test_reg, y_pred_reg) ** 0.5,
        r2_score(y_test_reg, y_pred_reg)
    ]
})
display(regression_test_metrics)

,Metric,Test score
0,MAE,2.230086
1,RMSE,3.402756
2,R²,0.377469


Step 12: Final model decision

The notebook selects the classification model with the highest mean cross-validation F1-score and the regression model with the lowest mean cross-validation MAE. The test set is then used once to check how each selected model performs on unseen data.

The model choice is made using cross-validation, while the test set gives the final, unbiased performance estimate. not just because its test score looks better after checking the test set.  The final output below states which models were selected and reports their test scores.